# Load and Test Azure AutoML Model (MLflow)

This notebook loads the MLflow model downloaded from an Azure Machine Learning registered model and runs test predictions.

**Model inputs (7 columns):**
- `Cost Center` (string)
- `Cost Element` (long)
- `Cost element name` (string)
- `Name` (string)
- `Purchase Order Text` (string)
- `Combine` (string)
- `Help_SAP` (string)

**Model output:** a string label (prediction).

## 1. Setup

Ensure the model folder (with `MLmodel`, `model.pkl`, `conda.yaml`) is in the **same directory as this notebook**.

In [1]:
# Optional: install required packages if not already present in your environment
# !pip install mlflow pandas numpy scikit-learn azureml-core azureml-automl-runtime

In [2]:
import os
import pandas as pd
import numpy as np
import mlflow

MODEL_DIR = os.getcwd()  # notebook runs in the folder that contains the model files

print("Model directory:", MODEL_DIR)
print("Files found:", [f for f in os.listdir(MODEL_DIR) if f in ("MLmodel", "model.pkl", "conda.yaml", "python_env.yaml")])

Model directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/onesvm/code/Users/onesinus231/downloaded_model
Files found: ['conda.yaml', 'MLmodel', 'model.pkl', 'python_env.yaml']


## 2. Load the Model

The model is an MLflow pyfunc model, so it can be loaded with `mlflow.pyfunc.load_model`.
It can also be loaded as a scikit-learn estimator via `mlflow.sklearn.load_model` (fallback).

In [3]:
# Load as MLflow pyfunc model
model = mlflow.pyfunc.load_model(MODEL_DIR)
print("Model loaded:", type(model))

2026/08/07 04:11:37 WARNING mlflow.utils.requirements_utils: Encountered an unexpected error (TypeError('expected string or bytes-like object')) while detecting model dependency mismatches. Set logging level to DEBUG to see the full traceback.


Model loaded: <class 'mlflow.pyfunc.PyFuncModel'>


In [4]:
# Optional: also load the raw underlying sklearn estimator
try:
    sk_model = mlflow.sklearn.load_model(MODEL_DIR)
    print("Underlying estimator type:", type(sk_model))
except Exception as e:
    print("Could not load sklearn flavor:", e)

Underlying estimator type: <class 'azureml.training.tabular.models.pipeline_with_ytransformations.PipelineWithYTransformations'>


## 3. Inspect Model Signature / Columns

In [5]:
from mlflow.models.signature import ModelSignature

signature = model.metadata.signature
print("Inputs:")
for s in signature.inputs.inputs:
    print(f"  - {s.name}  ({s.type})")
print("Outputs:", [str(o.type) for o in signature.outputs.inputs])

Inputs:
  - Cost Center  (DataType.string)
  - Cost Element  (DataType.long)
  - Cost element name  (DataType.string)
  - Name  (DataType.string)
  - Purchase Order Text  (DataType.string)
  - Combine  (DataType.string)
  - Help_SAP  (DataType.string)
Outputs: ['<U0']


## 4. Build Sample Test Data

Create a small sample DataFrame with the exact column names/types the model expects.
Replace the values below with your real test cases.

In [6]:
sample_df = pd.DataFrame([
    {
        "Cost Center": "1000",
        "Cost Element": 400000,
        "Cost element name": "Consumable material",
        "Name": "John Doe",
        "Purchase Order Text": "Purchase of raw material for production",
        "Combine": "1000-400000",
        "Help_SAP": "Cost center 1000, cost element 400000",
    },
    {
        "Cost Center": "2000",
        "Cost Element": 410000,
        "Cost element name": "External services",
        "Name": "Jane Smith",
        "Purchase Order Text": "External consulting services for Q3",
        "Combine": "2000-410000",
        "Help_SAP": "Cost center 2000, cost element 410000",
    },
])

# Ensure dtypes match the signature
sample_df["Cost Element"] = sample_df["Cost Element"].astype(np.int64)
sample_df["Cost Center"] = sample_df["Cost Center"].astype(str)
sample_df["Cost element name"] = sample_df["Cost element name"].astype(str)
sample_df["Name"] = sample_df["Name"].astype(str)
sample_df["Purchase Order Text"] = sample_df["Purchase Order Text"].astype(str)
sample_df["Combine"] = sample_df["Combine"].astype(str)
sample_df["Help_SAP"] = sample_df["Help_SAP"].astype(str)

sample_df.head()

,Cost Center,Cost Element,Cost element name,Name,Purchase Order Text,Combine,Help_SAP
0,1000,400000,Consumable material,John Doe,Purchase of raw material for production,1000-400000,"Cost center 1000, cost element 400000"
1,2000,410000,External services,Jane Smith,External consulting services for Q3,2000-410000,"Cost center 2000, cost element 410000"


## 5. Run Predictions

In [7]:
predictions = model.predict(sample_df)
print("Predictions:")
print(predictions)
print()
print("Prediction type:", type(predictions))

Predictions:
['Others Office_Equip_MICM' 'BUSINESS_TRAVEL_EXPENSE']

Prediction type: <class 'numpy.ndarray'>


In [8]:
# Combine inputs with predictions for a readable result
result = sample_df.copy()
result["Prediction"] = predictions
result

,Cost Center,Cost Element,Cost element name,Name,Purchase Order Text,Combine,Help_SAP,Prediction
0,1000,400000,Consumable material,John Doe,Purchase of raw material for production,1000-400000,"Cost center 1000, cost element 400000",Others Office_Equip_MICM
1,2000,410000,External services,Jane Smith,External consulting services for Q3,2000-410000,"Cost center 2000, cost element 410000",BUSINESS_TRAVEL_EXPENSE


## 6. Test With Your Own Data File

If you have a CSV file with the same columns (e.g. `test_data.csv`), uncomment and run the cells below.

In [17]:
# TEST_FILE = "test_data.csv"
# if os.path.exists(TEST_FILE):
#     test_df = pd.read_csv(TEST_FILE)
#     test_df["Cost Element"] = test_df["Cost Element"].astype(np.int64)
#     test_df = test_df.astype({"Cost Center": str, "Cost element name": str, "Name": str,
#                               "Purchase Order Text": str, "Combine": str, "Help_SAP": str})
#     print(test_df.shape)
#     test_df.head()
# else:
#     print(f"{TEST_FILE} not found in current directory.")

(10, 7)


In [19]:
# if os.path.exists("test_data.csv"):
#     test_preds = model.predict(test_df)
#     test_df["Prediction"] = test_preds
#     test_df.head(10)
#     test_df.to_csv("predictions_output.csv", index=False)

In [18]:
# test_df

,Cost Center,Cost Element,Cost element name,Name,Purchase Order Text,Combine,Help_SAP
0,1000,400000,Consumable material,John Doe,Purchase of raw material for production,1000-400000,"Cost center 1000, cost element 400000"
1,2000,410000,External services,Jane Smith,External consulting services for Q3,2000-410000,"Cost center 2000, cost element 410000"
2,3000,420000,Travel expenses,Maria Garcia,Airfare and hotel for business trip to Berlin,3000-420000,"Cost center 3000, cost element 420000"
3,4000,430000,Energy costs,Ahmed Hassan,Electricity bill for manufacturing plant,4000-430000,"Cost center 4000, cost element 430000"
4,5000,440000,Maintenance costs,Li Wei,Annual maintenance contract for HVAC system,5000-440000,"Cost center 5000, cost element 440000"
5,6000,450000,Software licenses,Sarah Johnson,Annual Microsoft 365 subscription renewal,6000-450000,"Cost center 6000, cost element 450000"
6,7000,460000,Training costs,David Brown,Employee training course in data analytics,7000-460000,"Cost center 7000, cost element 460000"
7,8000,470000,Office supplies,Priya Patel,Printer paper and toner cartridges,8000-470000,"Cost center 8000, cost element 470000"
8,9000,480000,Telecommunications,Michael Chen,Mobile phone bills for sales team,9000-480000,"Cost center 9000, cost element 480000"
9,10000,490000,Marketing expenses,Emma Wilson,Digital advertising campaign for new product l...,10000-490000,"Cost center 10000, cost element 490000"
